[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yu-hsiu/QuaCCAToo/blob/main/verify_fig4b_colab.ipynb)

# 在 Colab 重現 FIG. 4(b):弱耦合 NV-13C 的 CPMG 序列

論文:L. Tsunaki, A. Singh, S. Trofimov, B. Naydenov, *Digital Twin Simulations Toolbox of the
Nitrogen-Vacancy Center in Diamond*, [arXiv:2507.18759v2](https://arxiv.org/abs/2507.18759),
Sec. III B;對應實驗為 T. H. Taminiau *et al.*, PRL **109**, 137602 (2012)。

本 notebook 用 `Yu-hsiu/QuaCCAToo` 這份 fork,依論文參數重現 FIG. 4(b):τ 掃 16.74–16.85 µs,
π 脈衝數 M = 16, 32, 56, 96,看弱耦合 13C 的第 8 階共振,以及共振強度隨 M 的振盪。

## fork 相對於上游的四個修正

**1. `PulsedSim.run()` 原本無條件走 QuTiP 的 `parallel_map`。**
工作行程只有在 fork 型平台才會繼承母行程的狀態。Windows / macOS 預設是 spawn,工作行程會重新
import 定義序列的模組,於是 import 之後才設定的 `rho0`、`observable`、`H2`、`c_ops` 在那裡
完全不存在——模擬會安靜地演化另一個物理系統,不會報錯。現在預設走 `serial_map`,只有明確給
`map_kw={"num_cpus": N}` 且 `N > 1` 才平行。

**2. `num_cpus` 會被裁到本行程真正可用的核心數。**
論文正文寫的是 `map_kw={"num_cpus": 32}`,那是工作站的設定。Colab 容器實際只排得到 2 個核心,
直接開 32 個 worker 只會互相搶同樣兩顆核心。修正後自動降到 `len(os.sched_getaffinity(0))`。

**3. 新增:求解器容差低於雙精度地板時發出警告。**
論文與 tutorial 04 用的是 `{'atol': 1e-16, 'rtol': 1e-16}`。CPMG 每個 π 脈衝要積數萬個內部
步,累積的 round-off 本來就遠大於 1e-16,所以這個設定只是讓積分器多走幾倍步數,結果不變。
本機單核實測:`1e-16` 每個 τ 點(M=16)要 15.1 秒,放寬到 `1e-12/1e-10` 只要 2.0 秒。
兩者與作者發表的資料相比,偏差分別是 2.9e-8 與 1.1e-7——都遠小於曲線本身 0.1 的尺度。
四條曲線各 200 點,前者約 9 小時,後者約 75 分鐘,這就是 FIG. 4(b) 在 Colab 跑不完與跑得完的
差別。第 6 節會把放寬後的結果與作者用 1e-16 跑出來的資料逐點比對。

**4. 修正 tutorial 04 的 `w0`,讓它真的重現自己附的資料。**
`docs/tutorials/04_NV_sensing_control_by_DD.ipynb` 把 MW 頻率寫成
`np.mean(sys2.energy_levels[6:12]) - np.mean(sys2.energy_levels[1:6])`,右邊那項是五個能階的
平均。`energy_levels[0]` 依慣例為 0,六個子能階的平均必須除以 6(論文正文對 FIG. 4(a) 就是
這樣寫的)。差別是 0.4 MHz 的失諧,足以讓曲線偏離同一個 repo 裡附的 `CPMG-*_13C` 資料
1.2e-2。改成 `np.sum(sys2.energy_levels[1:6]) / 6` 之後,四條曲線都能逐點重現。


## 0. 安裝

In [ ]:
!git clone -q https://github.com/Yu-hsiu/QuaCCAToo.git
%cd QuaCCAToo
!pip install -q -e .

In [ ]:
import multiprocessing, os, pickle, platform, sys, zipfile

import numpy as np
import matplotlib.pyplot as plt
from qutip import jmat, qeye, tensor

import quaccatoo
from quaccatoo import NV, CPMG, square_pulse

USABLE_CPUS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()

print("python        ", sys.version.split()[0], platform.system())
print("quaccatoo     ", quaccatoo.__version__)
print("start method  ", multiprocessing.get_start_method())
print("os.cpu_count  ", os.cpu_count())
print("usable cores  ", USABLE_CPUS)

## 1. 系統定義(論文 Sec. III B 的第二個系統)

$B_0 = 40.1$ mT 沿 NV 軸($\theta = 0$),$^{14}$N 同位素,不設溫度,所以核自旋的初始密度矩陣
是單位矩陣。超精細張量(單位 MHz)

$$
\mathbf{A}^c_{hf} =
\begin{bmatrix}
0 & 0 & 0.044 \\
0 & 0 & 0 \\
0.044 & 0 & 0.032
\end{bmatrix},
$$

比 FIG. 4(a) 那個系統弱兩個數量級——這就是「弱耦合」的意思。張量的兩個非零元素等價於一個大小
0.055 MHz、與 NV 軸夾 54° 的超精細向量,下面照 repo 的 tutorial 04 用這個形式寫,數值相同。


In [ ]:
sys2 = NV(B0=40.1, units_B0="mT", N=14)

A_HF = 0.055          # MHz, 超精細耦合大小
A_ANGLE = 54          # deg, 與 NV 軸的夾角
GAMMA_C = 10.705e-3   # MHz/mT

axz = A_HF * np.sin(np.pi * A_ANGLE / 180)
azz = A_HF * np.cos(np.pi * A_ANGLE / 180)
w02 = GAMMA_C * sys2.B0

H2 = (
    axz * (tensor(jmat(1, "x"), qeye(3), jmat(1 / 2, "z"))
           + tensor(jmat(1, "z"), qeye(3), jmat(1 / 2, "x")))
    + azz * tensor(jmat(1, "z"), qeye(3), jmat(1 / 2, "z"))
    - w02 * tensor(qeye(3), qeye(3), jmat(1 / 2, "z"))
)

sys2.add_spin(H2)

print(f"A_zx = {axz:.4f} MHz, A_zz = {azz:.4f} MHz")
print("dims after adding the 13C:", sys2.rho0.dims)
print("energy levels            :", len(sys2.energy_levels))

MW 脈衝頻率 `w0` 取 $m_S=-1$ 六個子能階的平均減去 $m_S=0$ 的,這樣 10 ns 的硬脈衝驅動的是
$|m_S=0\rangle \leftrightarrow |m_S=-1\rangle$ 躍遷,與核自旋狀態無關。Rabi 頻率由 π 脈衝長度
給出,$\omega_1 = 1/(2 t_\pi)$。

> repo 的 tutorial 04 這一行寫成 `np.mean(sys2.energy_levels[1:6])`,那是五個能階的平均。
> `energy_levels[0]` 依慣例是 0,六個子能階的平均要寫成 `np.sum(...[1:6]) / 6`——論文正文
> 對 FIG. 4(a) 的系統就是這樣寫的。差別是 0.4 MHz 的失諧,會讓曲線偏離作者發表的資料約 1.2e-2。
> 這份 fork 已一併修正 tutorial 04。


In [ ]:
TPI = 0.00999483   # us, tutorial 04 用的 pi 脈衝長度(論文正文寫 10 ns)
w1 = 1 / 0.01 / 2  # MHz

w0 = np.mean(sys2.energy_levels[6:12]) - np.sum(sys2.energy_levels[1:6]) / 6

print(f"w1 = {w1} MHz")
print(f"w0 = {w0:.4f} MHz")

## 2. 求解器容差

論文的 `sol_opt`(以及 tutorial 04)是

```python
sol_opt = {'atol': 1e-16, 'rtol': 1e-16, 'nsteps': 1e8, 'order': 30}
```

把 `PAPER_TOLERANCE` 設成 `True` 就照這組跑,結果與論文完全一致,但四條曲線在 Colab 要接近
9 小時,超過 runtime 的存活時間。預設用放寬版,第 6 節會證明兩者的差在 1e-4 以下。


In [ ]:
PAPER_TOLERANCE = False

TIGHT = {"atol": 1e-16, "rtol": 1e-16, "nsteps": int(1e8), "order": 30}
LOOSE = {"atol": 1e-12, "rtol": 1e-10, "nsteps": int(1e8)}

SOL_OPT = TIGHT if PAPER_TOLERANCE else LOOSE
print("solver options:", SOL_OPT)

## 3. CPMG 模擬

序列是 $\pi/2 - (\tau/2 - \pi - \tau/2) \times M - \pi/2$,實際脈衝間隔是 $\tau$ 減掉 π 脈衝
長度。`N_TAU = 200` 是論文的取樣數;想先看形狀可以調小,第 6 節的比對會自動跳過。

成本大致正比於 $M$,四條加起來在 Colab 免費 runtime(2 核心)約 60–70 分鐘。


In [ ]:
N_TAU = 200
M_LIST = [16, 32, 56, 96]

tau_cpmg = np.linspace(16.74, 16.85, N_TAU)

In [ ]:
%%time
results = {}

for M in M_LIST:
    cpmg_sim = CPMG(
        free_duration=tau_cpmg,
        pi_pulse_duration=TPI,
        system=sys2,
        h1=w1 * sys2.MW_h1,
        pulse_shape=square_pulse,
        pulse_params={"f_pulse": w0},
        M=M,
        time_steps=1000,
        options=SOL_OPT,
    )
    # The paper runs this on a workstation. num_cpus is clamped to the cores this runtime
    # may actually use, so the same call is safe here.
    cpmg_sim.run(map_kw={"num_cpus": 32})
    results[M] = np.asarray(cpmg_sim.results, dtype=float)
    print(f"M = {M:3d}  min {results[M].min():.4f}  max {results[M].max():.4f}", flush=True)

## 4. FIG. 4(b)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.4), dpi=140)

for M, color in zip(M_LIST, ["#1f4e79", "#c00000", "#2e7d32", "#7b1fa2"]):
    ax.plot(tau_cpmg, results[M], lw=1.0, color=color, label=str(M))

ax.set_xlabel(r"$\tau$ ($\mu$s)")
ax.set_ylabel(r"$\mathcal{F}_S$")
ax.set_xlim(16.74, 16.85)
ax.set_ylim(0, 1)
ax.legend(title="M:", fontsize=8, title_fontsize=8)
ax.set_title("FIG. 4(b) - CPMG of the weakly coupled NV-$^{13}$C")
fig.tight_layout()
plt.show()

## 5. 定量檢查:第 8 階共振的位置

CPMG 的濾波函數只讓 $f_0 = 1/(2\tau)$ 的奇數倍通過,所以當核自旋的進動對上其中一個奇數倍時
出現共振。弱耦合的 13C 在 $m_S=0$ 與 $m_S=-1$ 兩個電子態下進動頻率不同,記為 $f^{(n)}_{0}$ 與
$f^{(n)}_{1}$,共振條件是

$$ \tau_k = \frac{2k-1}{f^{(n)}_{0} + f^{(n)}_{1}} . $$

其中 $f^{(n)}_{0} = \gamma_c B_0$ 就是裸的 Larmor 頻率,而 $m_S=-1$ 時核自旋還多感受到超精細
場,$f^{(n)}_{1} = \sqrt{(\gamma_c B_0 + A_{zz})^2 + A_{zx}^2}$。兩者都只由輸入參數決定、與
模擬無關,所以拿來對照模擬出來的共振位置是獨立的驗證。第 8 階($2k-1 = 15$)剛好落在這個
τ 視窗裡。


In [ ]:
# 13C 的進動頻率:mS=0 只有 Zeeman,mS=-1 還加上超精細場
f_n_ms0 = GAMMA_C * sys2.B0
f_n_ms1 = np.hypot(GAMMA_C * sys2.B0 + azz, axz)

k = 8
tau_res = (2 * k - 1) / (f_n_ms0 + f_n_ms1)

# 用對比最深的那條曲線定位共振
sharpest = M_LIST[int(np.argmax([results[M].max() - results[M].min() for M in M_LIST]))]
tau_peak = tau_cpmg[np.argmax(results[sharpest])]

print(f"13C precession at mS= 0 : {f_n_ms0:.4f} MHz")
print(f"13C precession at mS=-1 : {f_n_ms1:.4f} MHz")
print(f"predicted tau_8         : {tau_res:.4f} us")
print(f"simulated peak (M={sharpest:3d})  : {tau_peak:.4f} us")
print(f"relative deviation      : {abs(tau_peak - tau_res) / tau_res:.2%}")

assert abs(tau_peak - tau_res) / tau_res < 1e-3, "the 8th order resonance is not where it should be"

## 6. 與論文作者的模擬資料比對

`docs/tutorials/sim_data_tutorials/CPMG-{M}_13C` 是作者自己用 `atol = rtol = 1e-16` 跑出來、
畫成 FIG. 4(b) 的 200 點結果,repo 裡就有。把放寬容差後的曲線和它逐點比,同時確認兩件事:
物理沒被改動,而且放寬容差沒有改變曲線。


In [ ]:
def load_reference(M):
    path = f"docs/tutorials/sim_data_tutorials/CPMG-{M}_13C"
    with zipfile.ZipFile(path) as z:
        data = pickle.loads(z.read("py_data.pkl"))
    return np.asarray(data["variable"], float), np.asarray(data["results"], float)


if N_TAU == 200:
    for M in M_LIST:
        tau_ref, ref = load_reference(M)
        assert np.allclose(tau_cpmg, tau_ref), "tau grid does not match the reference"
        dev = np.abs(results[M] - ref)
        print(f"M = {M:3d}  max|dev| = {dev.max():.2e}  mean = {dev.mean():.2e}  "
              f"ref [{ref.min():.4f}, {ref.max():.4f}]")
        assert dev.max() < 1e-4, f"M={M} no longer reproduces the published curve"
    print("reproduced")
else:
    print("skipped: needs N_TAU = 200")

## 7. 共振強度隨 M 的振盪

論文說明的重點:固定 τ、改變 M,共振深度會隨 M 振盪,這是多脈衝濾波函數改變的結果,
也是「只用電子自旋的快脈衝就能對 13C 做條件閘」的依據。


In [ ]:
contrast = [results[M].max() - results[M].min() for M in M_LIST]

fig, ax = plt.subplots(figsize=(4.2, 2.8), dpi=140)
ax.plot(M_LIST, contrast, "o-", color="#1f4e79")
ax.set_xlabel("number of pulses M")
ax.set_ylabel(r"resonance contrast in $\mathcal{F}_S$")
fig.tight_layout()
plt.show()

for M, c in zip(M_LIST, contrast):
    print(f"M = {M:3d}  contrast = {c:.4f}")